# Colab Showcase 1: Source Ingestion and Provenance

This notebook is a reviewer-facing walkthrough of the ingestion layer behind the housing signals prototype.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Intuity-Technologies/hackathon/blob/main/output/jupyter-notebook/colab-template-01-data-ingestion.ipynb)


## What this notebook proves

- The project uses named ingestion scripts for official and curated housing inputs.
- Public readers can inspect the current source map without any credentials.
- Live ADLS reruns are still possible, but only through Colab user secrets.

### Current source snapshot from the checked-in demo artifacts

| Signal | Source | Period | Scope |
| --- | --- | --- | --- |
| Composite county housing pressure score | Signals parquet | 2025Q3 | County |
| County median sale price | Median sale price by county | 2025-02 | County |
| Regional homelessness adults | Homelessness report December 2025 | 2025-12 | Region |
| National housing cost overburden | Curated overburden parquet | 2024 | National |


In [ ]:
# Clone the repo in a fresh Colab runtime, then install the ingestion dependencies.
# !git clone https://github.com/Intuity-Technologies/hackathon.git
# %cd hackathon

!python --version
!pip install -q -r etl/requirements.txt


In [ ]:
import json
import os
from pathlib import Path

try:
    from google.colab import userdata
except ImportError:
    userdata = None


def load_secret(name: str, default: str = "") -> str:
    if userdata is None:
        return os.getenv(name, default)
    try:
        return userdata.get(name)
    except Exception:
        return os.getenv(name, default)


for key in ("AZURE_STORAGE_ACCOUNT", "AZURE_TENANT_ID", "AZURE_CLIENT_ID", "AZURE_CLIENT_SECRET"):
    value = load_secret(key)
    if value:
        os.environ[key] = value

os.environ.setdefault("RAW_FILE_SYSTEM", "raw")

manifest = json.loads(Path("data/demo/sources_manifest.json").read_text(encoding="utf-8"))
print("Sources in manifest:", len(manifest["sources"]))
print("Secret-backed live rerun ready:", all(os.getenv(key) for key in ("AZURE_STORAGE_ACCOUNT", "AZURE_TENANT_ID", "AZURE_CLIENT_ID", "AZURE_CLIENT_SECRET")))


In [ ]:
# Public mode: inspect the current checked-in source map.
for source in manifest["sources"][:5]:
    print(f"- {source['label']} | {source['source_name']} | {source['source_period']} | {source['coverage_scope']}")

# Live rerun mode: only execute when Colab user secrets are available.
if all(os.getenv(key) for key in ("AZURE_STORAGE_ACCOUNT", "AZURE_TENANT_ID", "AZURE_CLIENT_ID", "AZURE_CLIENT_SECRET")):
    print("Live ingestion rerun is enabled. Example commands:")
    print("!python etl/ingest/fetch_pxstat_table.py PEA08 --format CSV")
    print("!python etl/ingest/fetch_world_bank_indicator.py FP.CPI.TOTL.ZG --country IE")
    print('!python etl/ingest/fetch_central_bank_dataset.py --dataset-id retail_rates --url "https://<central-bank-url>.csv"')
else:
    print("Live ingestion cells are secret-gated. Add the Azure secrets in the Colab key panel to rerun them.")
